# Clean & Relabel — CIC-IDS-2018 raw CSV

Doc toan bo CSV goc tu `01_raw_data/raw_csv/` (tai truc tiep tu AWS S3
`s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/`), lam sach va chuan hoa nhan,
ghi ra:
- `clean_data/all_flows_clean.parquet` — ban tong hop (moi record 1 lan, dung cho EDA/tong quan).
- `clean_data/by_label/<Label>.parquet` — 1 file rieng cho moi trong 15 nhan chuan, dung lam
  input truc tiep cho `03_splits/build_leave_one_out_splits.py` (chi can gop/loai theo ten file,
  khong can filter DataFrame lon moi lan dung split).

**Khong doc lai bat ky file da qua xu ly truoc do** (khong dung
`UTS_Layer1/binary_split_dataset_z_Infilteration/`, khong dung `RAG_SHAP/data/Label_Flow/`) —
day la pipeline doc lap, tu lam sach tu nguon goc de audit duoc tung buoc cho paper.

**Luu y bo nho (du lieu lon, ~6.9GB CSV goc, uoc tinh hang trieu dong sau clean):**
xu ly va ghi output theo tung file CSV mot (streaming), **khong** `pd.concat()` toan bo 10 file
vao 1 DataFrame duy nhat truoc khi ghi — cach do doi hoi RAM bang ca 10 file cong lai cung luc.
Thay vao do: mo san moi `pyarrow.parquet.ParquetWriter` cho tung nhan + 1 cho ban tong hop, xu ly
tung CSV, `groupby("Label")` roi ghi ngay tung nhom vao dung writer, giai phong DataFrame cua file
do truoc khi doc file tiep theo. RAM dinh diem ~ kich thuoc 1 file CSV lon nhat (~4GB cho
Tuesday-20-02-2018), khong bao gio giu ca 10 file cung luc.

Xem `zeroshot_benchmark/plan.md` giai doan 2.

## 0. Setup

In [ ]:
import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

BASE_DIR = Path("..").resolve()

# Uu tien doc truc tiep tu Kaggle Dataset da mount (read-only, khong ton dung luong
# /kaggle/working/ va khong can download/giai nen lai) — fallback ve 01_raw_data/raw_csv
# khi chay local hoac khi dataset chua duoc add vao notebook.
_KAGGLE_MOUNT_DIR = Path("/kaggle/input/datasets/phttrnh/cic-ids-2018-raw-csv-zeroshot")
RAW_DIR = _KAGGLE_MOUNT_DIR if _KAGGLE_MOUNT_DIR.exists() else BASE_DIR / "01_raw_data" / "raw_csv"

CLEAN_DIR = Path(".").resolve() / "clean_data"
BY_LABEL_DIR = CLEAN_DIR / "by_label"
CLEAN_DIR.mkdir(exist_ok=True)
BY_LABEL_DIR.mkdir(exist_ok=True)

LABELS_CONFIG_PATH = BASE_DIR / "config" / "labels.json"
CLEANING_LOG_PATH = Path(".").resolve() / "cleaning_log.json"
OUTPUT_PARQUET_PATH = CLEAN_DIR / "all_flows_clean.parquet"

print("RAW_DIR:", RAW_DIR)
print("OUTPUT_PARQUET_PATH:", OUTPUT_PARQUET_PATH)
print("BY_LABEL_DIR:", BY_LABEL_DIR)

In [ ]:
with open(LABELS_CONFIG_PATH, "r", encoding="utf-8") as f:
    labels_config = json.load(f)

LABEL_MAP = labels_config["label_map"]
KNOWN_RAW_LABELS = set(LABEL_MAP.keys())
EXPECTED_STANDARD_LABELS = set(LABEL_MAP.values())

print(f"{len(LABEL_MAP)} nhan da biet trong config/labels.json")
print("Standard labels:", sorted(EXPECTED_STANDARD_LABELS))

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))
assert len(csv_files) > 0, f"Khong tim thay CSV nao trong {RAW_DIR} — chay download_from_aws.py truoc"
print(f"Tim thay {len(csv_files)} file CSV:")
for p in csv_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  {p.name}  ({size_mb:,.1f} MB)")

## 1. Ham lam sach 1 file CSV

Cac buoc lam sach (theo dung thu tu, ghi log so dong truoc/sau moi buoc):
1. Doc CSV (moi cot numeric ep ve numeric, loi -> NaN — bat header-lap luon o buoc nay).
2. Loai dong header-lap giua file (dong co gia tri cot `Label` == chuoi `"Label"` — dau hieu
   CICFlowMeter noi nhieu file con lai voi nhau, tung thay 25 dong nhu vay trong file test).
3. Loai dong co Inf o bat ky cot numeric nao (Flow Byts/s, Flow Pkts/s la 2 cot hay bi chia cho 0).
4. Loai dong co NaN o bat ky cot nao (sau khi ep numeric — bao gom NaN goc lan NaN sinh ra tu buoc 1).
5. Loai dong trung lap hoan toan trong pham vi file nay.
6. Chuan hoa ten cot `Label`: strip khoang trang, map qua `LABEL_MAP`. Neu gap nhan la vao
   danh sach da biet -> raise loi ngay (khong am tham bo qua/gan nham).

**Luu y**: `low_memory=False` trong `read_csv` bat buoc de tranh pandas doan sai dtype giua cac
chunk-doc-noi-bo cua 1 file lon (co the sinh warning/loi ngam khi 1 cot vua co gia tri numeric
vua co gia tri text o cac phan khac nhau cua file).

In [ ]:
def clean_single_file(csv_path: Path) -> tuple[pd.DataFrame, dict]:
    """Doc + lam sach 1 file CSV, tra ve (dataframe da sach, log-dict cho file nay)."""
    log = {"filename": csv_path.name}

    df = pd.read_csv(csv_path, low_memory=False)
    log["rows_raw"] = len(df)

    df.columns = [c.strip() for c in df.columns]
    assert "Label" in df.columns, f"{csv_path.name}: khong co cot 'Label'"

    # Mot so file (vd Thuesday-20-02-2018) co them 4 cot metadata dau (Flow ID, Src IP,
    # Src Port, Dst IP) so voi 9 file con lai — day la 2 dinh dang CICFlowMeter output
    # khac nhau trong chinh data goc AWS S3, khong phai loi tai/parse. Cac cot nay la
    # dinh danh flow (khong phai feature thong ke traffic), khong nam trong 78-feature
    # chuan (FEATURES trong UTS_Layer1/clad/data/dataset.py bat dau tu 'Dst Port') —
    # bo di de dong nhat schema giua 10 file, KHONG dua vao feature_cols/pd.to_numeric.
    metadata_cols = ["Flow ID", "Src IP", "Src Port", "Dst IP"]
    cols_to_drop_metadata = [c for c in metadata_cols if c in df.columns]
    if cols_to_drop_metadata:
        df = df.drop(columns=cols_to_drop_metadata)
        log["metadata_cols_dropped"] = cols_to_drop_metadata
    else:
        log["metadata_cols_dropped"] = []

    # Timestamp la text (vd '01/03/2018 08:47:38'), KHONG phai feature numeric.
    # Loai bo hoan toan khoi output - khong dua vao feature_cols de tranh bi ep
    # numeric nham thanh NaN (bug da phat hien qua smoke test: ep Timestamp numeric
    # lam 100% dong bi NaN o cot nay, dropna() xoa sach toan bo data).
    if "Timestamp" in df.columns:
        df = df.drop(columns=["Timestamp"])

    feature_cols = [c for c in df.columns if c != "Label"]

    n_before_header_dup = len(df)
    df = df[df["Label"].astype(str).str.strip() != "Label"].copy()
    log["dup_header_rows_dropped"] = n_before_header_dup - len(df)

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    n_before_inf = len(df)
    inf_mask = np.isinf(df[feature_cols].to_numpy(dtype="float64", na_value=0.0)).any(axis=1)
    df = df[~inf_mask].copy()
    log["inf_rows_dropped"] = n_before_inf - len(df)

    n_before_nan = len(df)
    df = df.dropna(subset=feature_cols + ["Label"]).copy()
    log["nan_rows_dropped"] = n_before_nan - len(df)

    n_before_dedup = len(df)
    df = df.drop_duplicates().copy()
    log["duplicate_rows_dropped"] = n_before_dedup - len(df)

    df["Label"] = df["Label"].astype(str).str.strip()
    raw_labels_seen = set(df["Label"].unique())
    unknown_labels = raw_labels_seen - KNOWN_RAW_LABELS
    assert not unknown_labels, (
        f"{csv_path.name}: gap nhan LA khong co trong config/labels.json: {unknown_labels}. "
        "Phai them vao label_map truoc khi chay tiep — khong duoc tu y bo qua."
    )

    log["label_counts_before_map"] = df["Label"].value_counts().to_dict()
    df["Label"] = df["Label"].map(LABEL_MAP)
    log["label_counts_after_map"] = df["Label"].value_counts().to_dict()

    log["rows_clean"] = len(df)
    return df, log

## 2. Xu ly streaming: doc tung file -> ghi ngay vao dung parquet writer

Mo san 1 `ParquetWriter` cho ban tong hop + 1 cho moi trong 15 nhan chuan **truoc khi** vong lap
bat dau (schema co dinh: 79 cot feature + `Label`, lay tu file CSV dau tien sau khi clean). Voi
moi file CSV: clean xong thi ghi thang vao writer tong hop, roi `groupby("Label")` ghi tung nhom
vao writer rieng cua nhan do. Xoa DataFrame cua file (`del df_clean`, `gc.collect()`) truoc khi
sang file tiep theo de giai phong RAM ngay, khong doi den cuoi vong lap.

In [ ]:
cleaning_log = {"generated_at": datetime.now().isoformat(), "files": {}}

combined_writer: pq.ParquetWriter | None = None
label_writers: dict[str, pq.ParquetWriter] = {}
label_row_counts: dict[str, int] = {lbl: 0 for lbl in EXPECTED_STANDARD_LABELS}
total_rows_written = 0
schema: pa.Schema | None = None

try:
    for csv_path in csv_files:
        print(f"Dang xu ly {csv_path.name} ...")
        df_clean, file_log = clean_single_file(csv_path)
        cleaning_log["files"][csv_path.name] = file_log
        print(
            f"  raw={file_log['rows_raw']:,}  "
            f"header_dup=-{file_log['dup_header_rows_dropped']:,}  "
            f"inf=-{file_log['inf_rows_dropped']:,}  "
            f"nan=-{file_log['nan_rows_dropped']:,}  "
            f"dup=-{file_log['duplicate_rows_dropped']:,}  "
            f"=> clean={file_log['rows_clean']:,}"
        )

        if schema is None:
            # Schema khoa lai tu file DAU TIEN. Cac file sau se bi EP ve dung schema
            # nay (khong chi combined_writer, ca group-writer theo label) - bat buoc,
            # vi cac file CSV khac nhau co the doc ra dtype khac nhau cho cung 1 cot
            # (int64 o file nay, float64 o file khac, tuy gia tri thuc te trong file
            # co xuat hien so thap phan hay khong). ParquetWriter.write_table() check
            # schema STRICT EQUALITY, khong tu cast - phai tu ep truoc khi ghi.
            table = pa.Table.from_pandas(df_clean, preserve_index=False)
            schema = table.schema
            combined_writer = pq.ParquetWriter(OUTPUT_PARQUET_PATH, schema)
            for lbl in EXPECTED_STANDARD_LABELS:
                label_writers[lbl] = pq.ParquetWriter(BY_LABEL_DIR / f"{lbl}.parquet", schema)
        else:
            table = pa.Table.from_pandas(df_clean, preserve_index=False, schema=schema)

        combined_writer.write_table(table)
        total_rows_written += len(df_clean)

        for lbl, group_df in df_clean.groupby("Label", observed=True):
            group_table = pa.Table.from_pandas(group_df, preserve_index=False, schema=schema)
            label_writers[lbl].write_table(group_table)
            label_row_counts[lbl] += len(group_df)

        del df_clean, table
        gc.collect()
finally:
    if combined_writer is not None:
        combined_writer.close()
    for w in label_writers.values():
        w.close()

print(f"\nHoan tat. Tong so dong da ghi (ban tong hop): {total_rows_written:,}")

## 3. Kiem tra tong quan (doc lai tu parquet, khong giu ban goc trong RAM)

In [ ]:
missing_standard_labels = sorted(lbl for lbl, cnt in label_row_counts.items() if cnt == 0)
assert not missing_standard_labels, (
    f"{len(missing_standard_labels)} nhan chuan khong co dong nao sau khi lam sach: "
    f"{missing_standard_labels}. Day la loi nghiem trong (vd file CSV bi lech schema khien "
    f"toan bo dong bi coerce NaN) — PHAI dieu tra va sua truoc khi dung data nay cho splits/training, "
    f"khong duoc bo qua."
)
print("OK: du 15 nhan chuan (Benign + 14 attack type) da xuat hien trong data.")

print("\nSo dong theo tung nhan (dem ngay trong luc ghi, khong can doc lai file):")
for lbl in sorted(label_row_counts):
    print(f"  {lbl:30s} {label_row_counts[lbl]:>12,}")

In [ ]:
# Kiem tra trung lap XUYEN SUOT nhieu file (vd cung 1 flow xuat hien o 2 file CSV khac nhau) —
# chi lam duoc bang cach doc lai theo tung nhan (nho hon nhieu so voi doc lai toan bo), vi buoc
# lam sach o tren chi loai duplicate TRONG PHAM VI 1 file, chua loai duplicate GIUA cac file.
cross_file_dup_total = 0
cross_file_dup_by_label = {}

for lbl in sorted(label_row_counts):
    if label_row_counts[lbl] == 0:
        continue
    df_lbl = pd.read_parquet(BY_LABEL_DIR / f"{lbl}.parquet")
    n_before = len(df_lbl)
    n_dup = df_lbl.duplicated().sum()
    cross_file_dup_by_label[lbl] = int(n_dup)
    cross_file_dup_total += int(n_dup)
    if n_dup > 0:
        df_lbl = df_lbl.drop_duplicates().reset_index(drop=True)
        df_lbl.to_parquet(BY_LABEL_DIR / f"{lbl}.parquet", index=False)
        label_row_counts[lbl] = len(df_lbl)
        print(f"  {lbl}: loai {n_dup:,} dong trung lap xuyen-file, con lai {len(df_lbl):,}")
    del df_lbl
    gc.collect()

print(f"\nTong so dong trung lap xuyen-file da loai: {cross_file_dup_total:,}")

if cross_file_dup_total > 0:
    print("CANH BAO: all_flows_clean.parquet (ban tong hop) van con cac dong trung lap xuyen-file")
    print("vi no duoc ghi truoc khi buoc nay chay — chi cac file by_label/*.parquet da duoc sua.")
    print("Neu can ban tong hop sach tuyet doi, doc lai tu by_label/*.parquet va ghep lai o buoc sau.")

## 4. Ghi cleaning log

In [ ]:
cleaning_log["summary"] = {
    "total_files": len(csv_files),
    "total_rows_written_combined": total_rows_written,
    "label_row_counts_final": label_row_counts,
    "cross_file_duplicates_dropped_by_label": cross_file_dup_by_label,
    "cross_file_duplicates_dropped_total": cross_file_dup_total,
    "missing_standard_labels": missing_standard_labels,
    "note_combined_file_may_contain_cross_file_duplicates": cross_file_dup_total > 0,
}

with open(CLEANING_LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaning_log, f, indent=2, ensure_ascii=False)

print(f"Da ghi cleaning log: {CLEANING_LOG_PATH}")

## 5. Xac nhan cuoi: doc lai vai file, kiem tra khop

In [ ]:
# Chi doc lai metadata (so dong) qua pyarrow, KHONG load toan bo data vao RAM — an toan voi file lon.
combined_meta = pq.ParquetFile(OUTPUT_PARQUET_PATH).metadata
print(f"all_flows_clean.parquet: {combined_meta.num_rows:,} dong, {combined_meta.num_columns} cot")
assert combined_meta.num_rows == total_rows_written, "So dong file tong hop khong khop voi log"

print("\nXac nhan tung file by_label:")
for lbl in sorted(label_row_counts):
    if label_row_counts[lbl] == 0:
        continue
    meta = pq.ParquetFile(BY_LABEL_DIR / f"{lbl}.parquet").metadata
    assert meta.num_rows == label_row_counts[lbl], f"{lbl}: so dong file khong khop voi log"
    print(f"  {lbl:30s} {meta.num_rows:>12,} dong  OK")

print("\nOK: toan bo file parquet (tong hop + theo nhan) da ghi va khop voi cleaning_log.json.")